# LeetCode #1333: Filter Restaurants by Vegan-Friendly, Price and Distance

https://leetcode.com/problems/filter-restaurants-by-vegan-friendly-price-and-distance/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: Filter then Sort ★** | $O(n \log n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
For each restaurant, scan all others to rank it by rating and ID while applying filters. Redundant comparisons make this $O(n^2)$.

### Optimal: Filter then Sort ★
First pass: discard any restaurant that fails the vegan, price, or distance constraint. Second pass: sort survivors by rating descending, then by `id` descending as a tiebreaker. Extract the IDs. $O(n \log n)$ dominated by sort; $O(n)$ space for the filtered list.

**Constraints:**
* $1 \leq \text{restaurants.length} \leq 10^4$
* Each restaurant: `[id, rating, veganFriendly, price, distance]`
* `veganFriendly` and `maxVeganFriendly` are 0 or 1
* $1 \leq \text{maxPrice}, \text{maxDistance} \leq 10^9$


## Solutions

### C#

In [ ]:
public class Solution {
    public IList<int> FilterRestaurants(int[][] restaurants, int veganFriendly, int maxPrice, int maxDistance) {
        return restaurants
            // Discard anything that violates any constraint
            .Where(r => (veganFriendly == 0 || r[2] == 1) && r[3] <= maxPrice && r[4] <= maxDistance)
            // Rating desc, then id desc to break ties deterministically
            .OrderByDescending(r => r[1])
            .ThenByDescending(r => r[0])
            .Select(r => r[0])
            .ToList();
    }
}

### Python

In [ ]:
class Solution:
    def filter_restaurants(
        self,
        restaurants: list[list[int]],
        vegan_friendly: int,
        max_price: int,
        max_distance: int
    ) -> list[int]:
        # Discard anything that violates any constraint
        kept = [
            r for r in restaurants
            if (vegan_friendly == 0 or r[2] == 1)
            and r[3] <= max_price
            and r[4] <= max_distance
        ]
        # Rating desc, then id desc to break ties deterministically
        kept.sort(key=lambda r: (r[1], r[0]), reverse=True)
        return [r[0] for r in kept]

### Go

In [ ]:
import "sort"

func filterRestaurants(restaurants [][]int, veganFriendly, maxPrice, maxDistance int) []int {
    var kept [][]int
    for _, r := range restaurants {
        // Discard anything that violates any constraint
        if (veganFriendly == 0 || r[2] == 1) && r[3] <= maxPrice && r[4] <= maxDistance {
            kept = append(kept, r)
        }
    }
    // Rating desc, then id desc to break ties deterministically
    sort.Slice(kept, func(i, j int) bool {
        if kept[i][1] != kept[j][1] { return kept[i][1] > kept[j][1] }
        return kept[i][0] > kept[j][0]
    })
    ids := make([]int, len(kept))
    for i, r := range kept {
        ids[i] = r[0]
    }
    return ids
}

### Rust

In [ ]:
impl Solution {
    pub fn filter_restaurants(
        mut restaurants: Vec<Vec<i32>>,
        vegan_friendly: i32,
        max_price: i32,
        max_distance: i32,
    ) -> Vec<i32> {
        // Discard anything that violates any constraint
        restaurants.retain(|r| {
            (vegan_friendly == 0 || r[2] == 1) && r[3] <= max_price && r[4] <= max_distance
        });
        // Rating desc, then id desc to break ties deterministically
        restaurants.sort_unstable_by(|a, b| {
            b[1].cmp(&a[1]).then(b[0].cmp(&a[0]))
        });
        restaurants.iter().map(|r| r[0]).collect()
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `restaurants = [[1,4,1,40,10],[2,8,0,50,5],[3,8,1,30,4]]`, `veganFriendly=1`, `maxPrice=50`, `maxDistance=10`
Filter removes restaurant 2 (not vegan). Survivors sorted by rating then id: `[3(8),1(4)]` → ids `[3,1]`.

### 2. Slightly Complex
**Input:** `veganFriendly=0` (don't filter by vegan), `maxPrice=30`, `maxDistance=3`
All three restaurants considered; only those with price ≤ 30 and distance ≤ 3 survive. Price/distance constraints act as independent gates — both must pass simultaneously.

### 3. Edge Case: Time Factor
**Input:** $n = 10{,}000$ restaurants, none filtered out
All 10,000 go to sort. $O(n \log n) \approx 130{,}000$ comparisons — the maximum sort workload. Filter step is $O(n)$ and free.

### 4. Edge Case: Space Factor
**Input:** $n = 10{,}000$ restaurants, all filtered out by `veganFriendly=1`
The `kept` list is empty. Sort operates on 0 elements. Output is an empty list — $O(1)$ extra space used beyond the (empty) result.

### 5. Almost-Impossible but Plausible
**Input:** Two restaurants with identical ratings: `[5,9,1,10,5]` and `[3,9,1,10,5]`
Both pass all filters. Tiebreaker is `id` descending: id 5 ranks before id 3 → `[5,3]`. Without the tiebreaker, output would be non-deterministic across language sort implementations.
